# Model Training

### Load data

In [1]:
import pandas as pd

train = pd.read_csv("../data/processed/train.csv")
test = pd.read_csv("../data/processed/test.csv")

In [2]:
X_train = train.drop("Churn", axis=1)
y_train = train["Churn"]

X_test = test.drop("Churn", axis=1)
y_test = test["Churn"]

### Model Evaluation Prep

In [3]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, probs)
    }

In [4]:
from pathlib import Path
import mlflow

project_root = Path.cwd().parent.resolve()

mlflow.set_tracking_uri(
    f"sqlite:///{project_root / 'mlflow.db'}"
)

experiment_name = "telco_churn_models"

artifact_location = f"file://{project_root / 'mlruns'}"

existing_experiment = mlflow.get_experiment_by_name(experiment_name)

if existing_experiment is None:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location
    )
else:
    experiment_id = existing_experiment.experiment_id

mlflow.set_experiment(experiment_name)

2026/06/16 09:50:32 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/16 09:50:32 INFO mlflow.store.db.utils: Updating database tables


<Experiment: artifact_location='file:///Users/haleighoeser/Documents/DataScience/Repos/predictflow-mlops/mlruns', creation_time=1781617833369, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781617833369, lifecycle_stage='active', name='telco_churn_models', tags={}, trace_location=None, workspace='default'>

### Model 1: Logistic Regression

In [7]:
import optuna
from sklearn.linear_model import LogisticRegression

mlflow.set_experiment("telco_churn_models")

def lr_objective(trial):

    params = {
        "C": trial.suggest_float("C", 0.001, 10.0, log=True),
        "penalty": trial.suggest_categorical("penalty", ["l2"]),
        "solver": "lbfgs"  # stable default for l2
    }

    with mlflow.start_run():

        model = LogisticRegression(
            **params, 
            max_iter=1000
        )

        model.fit(X_train, y_train)

        metrics = evaluate_model(model, X_test, y_test)

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.log_param("model", "logistic_regression")
        mlflow.log_param("mlflow_flavor", "sklearn")

        mlflow.log_metrics(metrics)

        mlflow.sklearn.log_model(model, "model")

        return metrics["f1"]

### Model 2: Random Forest

In [8]:
from sklearn.ensemble import RandomForestClassifier

def rf_objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
    }

    with mlflow.start_run():

        model = RandomForestClassifier(
            **params,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        metrics = evaluate_model(model, X_test, y_test)

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.log_param("model", "random_forest")
        mlflow.log_param("mlflow_flavor", "sklearn")

        mlflow.sklearn.log_model(model, "model")

        return metrics["f1"]

### Model 3: XGBoost

In [9]:
from xgboost import XGBClassifier

scale_pos_weight = (
    len(y_train[y_train == 0]) /
    len(y_train[y_train == 1])
)

def xgb_objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }

    with mlflow.start_run():

        model = XGBClassifier(
            **params,
            eval_metric="logloss",
            scale_pos_weight = scale_pos_weight,
            random_state=42
        )

        model.fit(X_train, y_train)

        metrics = evaluate_model(model, X_test, y_test)

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.log_param("model", "xgboost")
        mlflow.log_param("mlflow_flavor", "xgboost")

        mlflow.xgboost.log_model(model, artifact_path="model")

        return metrics["f1"]

### Model 4: LightGBM

In [10]:
from lightgbm import LGBMClassifier

def lgb_objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
    }

    with mlflow.start_run():

        model = LGBMClassifier(
            **params,
            class_weight="balanced",
            random_state=42
        )

        model.fit(X_train, y_train)

        metrics = evaluate_model(model, X_test, y_test)

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.log_param("model", "lightgbm")
        mlflow.log_param("mlflow_flavor", "lightgbm")

        mlflow.lightgbm.log_model(model, artifact_path="model")

        return metrics["f1"]

### Executing Model Training and Evaluation

In [11]:
study_lr = optuna.create_study(direction="maximize")
study_lr.optimize(lr_objective, n_trials=10)

[I 2026-06-16 09:50:46,140] A new study created in memory with name: no-name-f61573e7-2b5a-4f2d-b909-96bff297e957
/Users/haleighoeser/Documents/DataScience/Repos/predictflow-mlops/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/haleighoeser/Documents/DataScience/Repos/predictflow-mlops/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also

In [12]:
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(rf_objective, n_trials=10)

[I 2026-06-16 09:51:12,811] A new study created in memory with name: no-name-cbb41e07-ec3d-4500-b460-06c1816a3e9d
2026/06/16 09:51:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 09:51:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-06-16 09:51:15,660] Trial 0 finished with value: 0.5008291873963516 and parameters: {'n_estimators': 300, 'max_depth': 5, 'min_samples_split': 11, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.5008291873963516.
2026/06/16 09:51:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 09:51:16 WARNING mlflow.sklearn: 

In [13]:
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(xgb_objective, n_trials=50)

[I 2026-06-16 09:51:42,962] A new study created in memory with name: no-name-7268bc45-8545-4d38-8b8e-44bfc36cfa17
2026/06/16 09:51:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[I 2026-06-16 09:51:46,888] Trial 0 finished with value: 0.6225769669327252 and parameters: {'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.050797201007984745, 'subsample': 0.9834858201467147, 'colsample_bytree': 0.9900757200001984}. Best is trial 0 with value: 0.6225769669327252.
2026/06/16 09:51:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[I 2026-06-16 09:51:50,930] Trial 1 finished with value: 0.5533596837944664 and parameters: {'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.14168450015095027, 'subsample': 0.7058363838639851, 'colsample_bytree': 0.8446795092725808}. Best is trial 0 with value: 0.6225769669327252.
2026/06/16 09:51:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use 

In [14]:
study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(lgb_objective, n_trials=50)

[I 2026-06-16 09:55:00,299] A new study created in memory with name: no-name-4c552ce5-4467-4fad-84f6-fd61b803b544


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4130
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000886 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 623
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


2026/06/16 09:55:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 09:55:01 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-06-16 09:55:05,339] Trial 0 finished with value: 0.6203904555314533 and parameters: {'n_estimators': 600, 'num_leaves': 20, 'learning_rate': 0.037081610929028744, 'min_child_samples': 64}. Best is trial 0 with value: 0.6203904555314533.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4130
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000539 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 623
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

2026/06/16 09:55:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

2026/06/16 10:00:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 10:00:12 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-06-16 10:00:15,939] Trial 47 finished with value: 0.6002430133657352 and parameters: {'n_estimators': 500, 'num_leaves': 94, 'learning_rate': 0.0278319849261917, 'min_child_samples': 27}. Best is trial 29 with value: 0.6261780104712041.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4130
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000713 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 623
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


2026/06/16 10:00:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 10:00:18 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-06-16 10:00:22,741] Trial 48 finished with value: 0.6193118756936737 and parameters: {'n_estimators': 600, 'num_leaves': 40, 'learning_rate': 0.01933613224874462, 'min_child_samples': 57}. Best is trial 29 with value: 0.6261780104712041.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4130
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000811 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 623
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


2026/06/16 10:00:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 10:00:24 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-06-16 10:00:28,324] Trial 49 finished with value: 0.630457933972311 and parameters: {'n_estimators': 800, 'num_leaves': 18, 'learning_rate': 0.016281139883662272, 'min_child_samples': 68}. Best is trial 49 with value: 0.630457933972311.


### Model Selection

In [15]:
import mlflow
import pandas as pd

experiment = mlflow.get_experiment_by_name("telco_churn_models")

runs = mlflow.search_runs([experiment.experiment_id])

best_run = runs.sort_values("metrics.f1", ascending=False).iloc[0]

best_run_id = best_run["run_id"]
best_run

run_id                                       e4807dddec6c43b08e6d5b926a955b00
experiment_id                                                               1
status                                                               FINISHED
artifact_uri                file:///Users/haleighoeser/Documents/DataScien...
start_time                                   2026-06-16 14:00:22.750000+00:00
end_time                                     2026-06-16 14:00:28.320000+00:00
metrics.f1                                                           0.630458
metrics.accuracy                                                     0.753376
metrics.roc_auc                                                      0.835334
params.n_estimators                                                       800
params.num_leaves                                                          18
params.model                                                         lightgbm
params.mlflow_flavor                                            

In [17]:
model_uri = f"runs:/{best_run_id}/model"

flavor = best_run['params.mlflow_flavor']

if flavor == "sklearn":
    model = mlflow.sklearn.load_model(model_uri)

elif flavor == "xgboost":
    model = mlflow.xgboost.load_model(model_uri)

elif flavor == "lightgbm":
    model = mlflow.lightgbm.load_model(model_uri)

# best_model = mlflow.sklearn.load_model(model_uri)

### Register Best Model

In [18]:
mlflow.register_model(
    model_uri,
    name="telco_churn_best_model"
)

Successfully registered model 'telco_churn_best_model'.
2026/06/16 10:14:01 WARNING mlflow.tracking._model_registry.fluent: Run with id e4807dddec6c43b08e6d5b926a955b00 has no artifacts at artifact path 'model', registering model based on models:/m-aa4ca5d3e35c4687a1225d04de5dbb8e instead
Created version '1' of model 'telco_churn_best_model'.


<ModelVersion: aliases=[], creation_timestamp=1781619241299, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1781619241299, metrics=None, model_id=None, name='telco_churn_best_model', params=None, run_id='e4807dddec6c43b08e6d5b926a955b00', run_link=None, source='models:/m-aa4ca5d3e35c4687a1225d04de5dbb8e', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>